<a href="https://colab.research.google.com/github/benarjii/miniproject_sem6/blob/main/Evaluating_RAG_Pipelines_o3_mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install haystack-ai
!pip install "datasets>=2.6.1"
!pip install "sentence-transformers>=3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.6/451.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires

In [ ]:
from haystack.telemetry import tutorial_running

tutorial_running(35)

In [ ]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("vblagoje/PubMedQA_instruction", split="train")
dataset = dataset.select(range(1000))
all_documents = [Document(content=doc["context"]) for doc in dataset]
all_questions = [doc["instruction"] for doc in dataset]
all_ground_truth_answers = [doc["response"] for doc in dataset]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/498 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/986k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/272458 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from typing import List
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy

document_store = InMemoryDocumentStore()

document_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
document_writer = DocumentWriter(document_store=document_store, policy=DuplicatePolicy.SKIP)

indexing = Pipeline()
indexing.add_component(instance=document_embedder, name="document_embedder")
indexing.add_component(instance=document_writer, name="document_writer")

indexing.connect("document_embedder.documents", "document_writer.documents")

indexing.run({"document_embedder": {"documents": all_documents}})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

{'document_writer': {'documents_written': 1000}}

In [ ]:
import os
from getpass import getpass
from haystack.components.builders import AnswerBuilder, ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")

template = [
    ChatMessage.from_user(
        """
        You have to answer the following question based on the given context information only.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}

        Question: {{question}}
        Answer:
        """
    )
]

rag_pipeline = Pipeline()
rag_pipeline.add_component(
    "query_embedder", SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
rag_pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
rag_pipeline.add_component("generator", OpenAIChatGenerator(model="o3-mini-2025-01-31"))
rag_pipeline.add_component("answer_builder", AnswerBuilder())

rag_pipeline.connect("query_embedder", "retriever.query_embedding")
rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "generator.messages")
rag_pipeline.connect("generator.replies", "answer_builder.replies")
rag_pipeline.connect("retriever", "answer_builder.documents")

Enter OpenAI API key:··········


🚅 Components
  - query_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - generator: OpenAIChatGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - retriever.documents -> answer_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.messages (List[ChatMessage])
  - generator.replies -> answer_builder.replies (List[ChatMessage])

In [ ]:
question = "Do high levels of procalcitonin in the early phase after pediatric liver transplantation indicate poor postoperative outcome?"

response = rag_pipeline.run(
    {
        "query_embedder": {"text": question},
        "prompt_builder": {"question": question},
        "answer_builder": {"query": question},
    }
)
print(response["answer_builder"]["answers"][0].data)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Yes, high early postoperative procalcitonin levels are associated with poor outcomes after pediatric liver transplantation. According to the study, patients with elevated procalcitonin on postoperative day 2 experienced higher International Normalized Ratio values on day 5, an increased incidence of primary graft non-function, and longer stays in the pediatric intensive care unit and on mechanical ventilation. These findings indicate that high procalcitonin levels signal a worse postoperative prognosis.


In [ ]:
import random

questions, ground_truth_answers, ground_truth_docs = zip(
    *random.sample(list(zip(all_questions, all_ground_truth_answers, all_documents)), 25)
)

In [ ]:
rag_answers = []
retrieved_docs = []

for question in list(questions):
    response = rag_pipeline.run(
        {
            "query_embedder": {"text": question},
            "prompt_builder": {"question": question},
            "answer_builder": {"query": question},
        }
    )
    print(f"Question: {question}")
    print("Answer from pipeline:")
    print(response["answer_builder"]["answers"][0].data)
    print("\n-----------------------------------\n")

    rag_answers.append(response["answer_builder"]["answers"][0].data)
    retrieved_docs.append(response["answer_builder"]["answers"][0].documents)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Is ulcerative proctitis a frequent location of paediatric-onset UC and not a minor disease : a population-based study?
Answer from pipeline:
Based on the study, while ulcerative proctitis (UP) represented 25% of paediatric‐onset UC cases at diagnosis, it was not a benign or minor form of the disease. Nearly half of these patients experienced colonic extension over time—with cumulative risks reaching 45% at 5 years and 52% at 10 years—and their outcomes (including risks for needing anti-TNF‐α therapy and colectomy) were similar to those with more extensive disease. Thus, even though UP may seem limited at onset, its progression and therapeutic requirements indicate that it is far from being a minor disease.

-----------------------------------



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Does hypercholesterolemia increase the production of leukotriene B4 in neutrophils by enhancing the nuclear localization of 5-lipoxygenase?
Answer from pipeline:
Yes, the data indicate that hypercholesterolemia increases leukotriene B4 production in neutrophils through mechanisms that include enhanced nuclear localization of 5-lipoxygenase. Although the total levels of 5-LO remain similar, hypercholesterolemia is associated with increased nuclear localization and phosphorylation of 5-LO (as well as ERK1/2), which promotes LTB4 synthesis.

-----------------------------------



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Question: Does cardiac-specific ablation of synapse-associated protein SAP97 in mice decrease potassium currents but not sodium current?
Answer from pipeline:
Yes, cardiac-specific ablation of SAP97 in mice led to a decrease in potassium currents (specifically IK1, Ito, and IKur) without affecting the sodium current (INa).

-----------------------------------



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Is pattern of first recurrent lesions in pancreatic cancer : hepatic relapse associated with dismal prognosis and portal vein invasion?
Answer from pipeline:
Yes, based on the context hepatic relapse is linked to a dismal prognosis—it showed significantly shorter overall survival and was an independent prognostic factor (p<0.001). Moreover, pathological portal vein invasion was identified as the only independent risk factor for hepatic relapse (p=0.045).

-----------------------------------



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Is eGFR expression associated with poor outcome in cutaneous squamous cell carcinoma?
Answer from pipeline:
Based on the provided context about cutaneous squamous cell carcinoma (CSCC), EGFR overexpression in the primary tumors was found to be associated with lymph node progression, TNM stage progression, and increased proliferation (as indicated by Ki-67 staining). Furthermore, EGFR overexpression along with a poor grade of differentiation were identified as the strongest independent variables defining lymph node metastasis and clinical progression in CSCC. 

Thus, yes, overexpression of EGFR is associated with a poor outcome in CSCC.

-----------------------------------



While each evaluator is a component that can be run individually in Haystack, they can also be added into a pipeline. This way, we can construct an `eval_pipeline` that includes all evaluators for the metrics we want to evaluate our pipeline on.

In [ ]:
from haystack.components.evaluators.document_mrr import DocumentMRREvaluator
from haystack.components.evaluators.faithfulness import FaithfulnessEvaluator
from haystack.components.evaluators.sas_evaluator import SASEvaluator

eval_pipeline = Pipeline()
eval_pipeline.add_component("doc_mrr_evaluator", DocumentMRREvaluator())
eval_pipeline.add_component("faithfulness", FaithfulnessEvaluator())
eval_pipeline.add_component("sas_evaluator", SASEvaluator(model="sentence-transformers/all-MiniLM-L6-v2"))

results = eval_pipeline.run(
    {
        "doc_mrr_evaluator": {
            "ground_truth_documents": list([d] for d in ground_truth_docs),
            "retrieved_documents": retrieved_docs,
        },
        "faithfulness": {
            "questions": list(questions),
            "contexts": list([d.content] for d in ground_truth_docs),
            "predicted_answers": rag_answers,
        },
        "sas_evaluator": {"predicted_answers": rag_answers, "ground_truth_answers": list(ground_truth_answers)},
    }
)

100%|██████████| 25/25 [01:08<00:00,  2.76s/it]


### Constructing an Evaluation Report

Once we've run our evaluation pipeline, we can also create a full evaluation report. Haystack provides an `EvaluationRunResult` which we can use to display an `aggregated_report` 👇

In [ ]:
from haystack.evaluation.eval_run_result import EvaluationRunResult

inputs = {
    "question": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "answer": list(ground_truth_answers),
    "predicted_answer": rag_answers,
}

evaluation_result = EvaluationRunResult(run_name="pubmed_rag_pipeline", inputs=inputs, results=results)
evaluation_result.aggregated_report()

{'metrics': ['doc_mrr_evaluator', 'faithfulness', 'sas_evaluator'],
 'score': [1.0, np.float64(1.0), np.float64(0.7191868603229523)]}

#### Extra: You can also see a detailed report with the scores for each sample in your dataset, and we will choose the output format as DataFrame

In [ ]:
results_df = evaluation_result.detailed_report(output_format='df')
results_df

,question,contexts,answer,predicted_answer,doc_mrr_evaluator,faithfulness,sas_evaluator
0,Is ulcerative proctitis a frequent location of...,[Natural history of paediatric-onset ulcerativ...,UP is frequent in paediatric-onset UC and shou...,"Based on the study, while ulcerative proctitis...",1.0,1.0,0.754055
1,Does hypercholesterolemia increase the product...,[Neutrophils can synthesize leukotriene B4 (LT...,Hypercholesterolemia increases LTB4 production...,"Yes, the data indicate that hypercholesterolem...",1.0,1.0,0.880061
2,Does ultrasound elastography correlate treatme...,[To investigate the relationship between tissu...,"With a few exceptions, SVR improved LS. All pa...","Yes, ultrasound elastography does correlate wi...",1.0,1.0,0.454401
3,Is pseudomonas aeruginosa in CF and non-CF hom...,[For patients with cystic fibrosis (CF) Pseudo...,These findings implicate drains as important p...,"Yes, according to the context, Pseudomonas aer...",1.0,1.0,0.687885
4,Does preoperative anemia increase postoperativ...,[Preoperative anemia may affect postoperative ...,Preoperative anemia in elective cranial neuros...,"Yes, preoperative anemia does increase postope...",1.0,1.0,0.895171
5,Does loss of Frzb and Sfrp1 differentially aff...,[To investigate the specific role of Frizzled-...,"Using the DMM model, we demonstrated that FRZB...",Yes. The study found that deleting Frzb and Sf...,1.0,1.0,0.693589
6,Does extracellular vesicle-driven information ...,[Continuous exposure to particulate air pollut...,The study's findings support the hypothesis th...,"Based on the provided context, yes. The study ...",1.0,1.0,0.610129
7,Is reduction of methicillin-resistant Staphylo...,[Antibiotic resistance is a challenge in long-...,On-site MRSA surveillance with targeted decolo...,Answer: Yes. The study demonstrated that using...,1.0,1.0,0.739027
8,Does bile deficiency induce changes in intesti...,[Biliary tract obstruction is a common clinica...,Bile deficiency in the intestine upregulates t...,"Yes, bile deficiency (as induced by biliary du...",1.0,1.0,0.851983
9,Does computer simulation of lumbar flexion sho...,[The lumbar facet capsular ligament (FCL) is a...,We found that in-plane and through-plane shear...,"Yes, the simulation indicated that shear is in...",1.0,1.0,0.664086


Having our evaluation results as a dataframe can be quite useful. For example, below we can use the pandas dataframe to filter the results to the top 3 best scores for semantic answer similarity (`sas_evaluator`) as well as the bottom 3 👇


In [ ]:
# o3-mini
import pandas as pd

top_3 = results_df.nlargest(3, "sas_evaluator")
bottom_3 = results_df.nsmallest(3, "sas_evaluator")
pd.concat([top_3, bottom_3])

,question,contexts,answer,predicted_answer,doc_mrr_evaluator,faithfulness,sas_evaluator
4,Does preoperative anemia increase postoperativ...,[Preoperative anemia may affect postoperative ...,Preoperative anemia in elective cranial neuros...,"Yes, preoperative anemia does increase postope...",1.0,1.0,0.895171
1,Does hypercholesterolemia increase the product...,[Neutrophils can synthesize leukotriene B4 (LT...,Hypercholesterolemia increases LTB4 production...,"Yes, the data indicate that hypercholesterolem...",1.0,1.0,0.880061
11,Do nomograms incorporating serum C-reactive pr...,[To incorporate C-reactive protein into nomogr...,We have generated nomograms incorporating seru...,"Yes, the nomograms that incorporate serum C-re...",1.0,1.0,0.852763
2,Does ultrasound elastography correlate treatme...,[To investigate the relationship between tissu...,"With a few exceptions, SVR improved LS. All pa...","Yes, ultrasound elastography does correlate wi...",1.0,1.0,0.454401
18,Is the transposable element environment of hum...,[Only 2 % of the human genome code for protein...,"Taken together, these results suggest that the...","Yes, the transposable element environment is a...",1.0,1.0,0.540590
13,Is tumor-related leukocytosis associated with ...,[To evaluate response to radiation and clinica...,This study reveals the aggressive nature of ce...,"Yes. In the study, patients with tumor-related...",1.0,1.0,0.565599
